## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Set parent directory.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import pathlib
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta

# Set data directory and base directory
base_dir = Path.cwd()

Download repo and the hg38 fasta zipped file.

In [4]:
# Download repo to access scripts
!git clone https://github.com/eddykang06/mini-gLM.git
&cd mini-gLM

# Add repo scripts to path
sys.path.append("/content/mini-gLM-src")

# Download zipped human genome files
!mkdir -p data
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz" -o data/hg38.fa.gz
!curl -L -C - "https://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes" -o data/hg38.chrom.sizes
!gunzip -c data/hg38.fa.gz > data/hg38.fa # Unzip to fasta

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  938M  100  938M    0     0  63.9M      0  0:00:14  0:00:14 --:--:-- 60.9M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11672  100 11672    0     0  93291      0 --:--:-- --:--:-- --:--:-- 94129


Sample random sections of the genome, weighted by chromosome length and within the specified sequence length range.

In [9]:
from glm_pipeline import sample_from_fasta

# Set seed
rng = np.random.default_rng(111)

# Data directory
data_dir = base_dir / "data"
fasta_file = data_dir / "hg38.fa"
chrom_size_file = data_dir / "hg38.chrom.sizes"

# Set parameters
n_seqs = 100
min_length = 20
max_length = 500

# Get pretraining data
pretraining = sample_from_fasta(
    fasta_file = fasta_file,
    chrom_size_file = chrom_size_file,
    n_seqs = n_seqs,
    min_length = min_length,
    max_length = max_length
)

## Tokenizer

Steps to build tokenizer:
- Break corpus into words (["A", "C", "G", "T"])
- BPE algorithm

In [ ]:
def find_all_adjacent_pairs(char_list):
    """
    Given a list of characters, return a dictionary with keys as unique adjacent character tuples and values as the # of times 
    that adjacent pair appears in the list

    Ex: ["A", "C", "G"] -> {("A", "C"): 1, ("C", "G"): 1}
    """
    all_pairs = []

    for i in range(len(char_list) - 1):

        pair = "".join(char_list[i:i+2])
        all_pairs.append(pair)

    unique_pairs, counts = np.unique_counts(all_pairs)
    unique_pair_counts = {tuple(pair): int(count) for pair, count in zip(unique_pairs, counts)}

    return unique_pair_counts

In [46]:
dict = {
    "a":1,
    "b":2
}
dict.values() == np.max(dict.values())

False

In [40]:
arr = [0, 1, 2, 2, 2]
np.argmax(arr)

np.int64(2)

Quick BPE implementation.

In [ ]:
# Set seed for random selection
seed = 111
rng = np.default_rng(seed)

# Sample corpus
corpus = [
    "ACGTTGA",
    "ACGTACA",
    "AATTG"
]

# Initial vocab
vocab = ["A", "C", "G", "T"]

# Set size of interest
final_size = 10

# Store merge rules for future tokenization
merge_rules = {}

# In iteration 1, simple split each sequence into list (treat each sequence as a word)
split_corpus = [list(seq) for seq in corpus]

# Iterate until vocab reaches the specified length
while len(vocab) < final_size:

    # Dictionary to store the counts for each pair seen in the corpus
    pair_counts_all = {}

    # For each sequence in the corpus, find the unique pairs and store them a lists
    for seq in split_corpus:

        # Find counts of all adjacent pairs in sequence and store as a dictionary
        pair_counts_seq = find_all_adjacent_pairs(seq)

        for pair, count in pair_counts_all.items():
            
            # If the pair hasn't been seen yet, add it as a new key, value pair
            if pair not in pair_counts_all.keys():
                pair_counts_all[pair] = count

            # If the pair has already been seen, simply update the key
            else:
                pair_counts_all[pair] += count

    # Store pairs and counts
    pairs = np.array(list(pair_counts_all.keys()))
    counts = np.array(list(pair_counts_all.values))

    # Find pair(s) with the highest counts
    best_idx = np.argwhere(counts == counts.max())

    # If multiple pairs are found, then randomly select one
    if len(best_idx) > 1:
        random_idx = np.random.choice(best_idx)
        best_pair = pairs(random_idx)
    
    else:
        best_pair = pairs(best_idx)

    # Join all instances of the pair in split_corpus
    # THIS IS THE HARD PART**

    # Add new token to vocbaulary
    new_token = "".join(best_pair)
    vocab.append(new_token)

    # Store merge rule
    merge_rule = {best_pair: new_token}
    merge_rules.update(merge_rule)

print(vocab)

TypeError: dict.keys() takes no arguments (1 given)

In [ ]:
def count_motifs(seq_list: list | np.array, motif : str) -> int:
    """
    Function to count the number of times that a DNA motif occurs in a list of sequences

    Args:
        seq_list : List of DNA sequences as strings composed of "ACGT"
        motif    : DNA motif as str
    
    Returns:
        counts : # of times that the motif occurs in the list of sequences
    """
    counts = 0

    for seq in seq_list:
        

def implement_bpe(corpus: list | np.array, desired_size: int) -> np.array:
    """
    Function to implement byte-pair encoding algorithm to a corpus of DNA sequences until a desired vocab size is reached

    Args:
        corpus       : List of DNA sequences as strings composed of "ACGT"
        desired_size : Int describing the desired final vocabulary size

    Returns:
        vocab : Array storing the final vocab
    """
    # Initial vocab consists of 4 DNA bases
    vocab = ["A", "C", "G", "T"]

    while len(vocab) < desired_size:
        
        # Compute all possible token pairs
        pairs = 

        for token in vocab:

            # Count the number of times the token appears in the corpus

            # Find the two highest frequencies

            # Merge these tokens

            # Add this new token to the vocabulary

    return vocab


In [ ]:
def seqs_to_corpus():

    
    return corpus



class BPE_Tokenizer():
    def __init__(self, vocab, vocab_size):
        self.vocab = ["A", "C", "G", "T"]
        self.vocab_size = 4
    
    # Fit tokenizer on list of sequences
    def fit(self, seqs, desired_size):

    
    # Tokenize a given sequence
    def tokenize(self, seqs):




In [ ]:
class object:
    def __init__(self, stat):
        self.stat = stat



Reference: https://huggingface.co/learn/llm-course/en/chapter6/5

## Model

In [ ]:
!pip install accelerate
!pip install datasets
!pip install transformers
!pip install torch
!pip install flash-attn

import accelerate
import flash_attn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

torch.backends.cudnn.benchmark=True

Implement a sparse attention transformer.

Implement the model with 10 transformer blocks.

In [ ]:
class miniglm(nn.Module):

    def __init__(self):